# 📊 CLM (Contrastive Language Model) - Comprehensive Evaluation Report

## Mục lục
1. [Giới thiệu CLM](#1-giới-thiệu-clm)
2. [Cài đặt và Khởi tạo](#2-cài-đặt-và-khởi-tạo)
3. [Tải Dataset](#3-tải-dataset)
4. [Inference Engine](#4-inference-engine)
5. [Đánh giá toàn diện](#5-đánh-giá-toàn-diện)
6. [Trực quan hóa kết quả](#6-trực-quan-hóa-kết-quả)
7. [Phân tích thống kê](#7-phân-tích-thống-kê)
8. [Xuất báo cáo](#8-xuất-báo-cáo)

---

## 1. Giới thiệu CLM

### 1.1 CLM là gì?

**CLM (Contrastive Language Model)** là một "System One" model - được thiết kế cho việc ra quyết định nhanh và có khả năng tổng quát hóa cao.

#### Kiến trúc:
```
Input (State) ──▶ Qwen3-8B (frozen) ──▶ Projection Heads ──▶ Scores
                    +                   (20M params)
Candidate Tools ──▶ Qwen3-8B (frozen) ──┘
```

#### Điểm khác biệt so với LLM truyền thống:

| Aspect | LLM truyền thống | CLM |
|--------|------------------|-----|
| Phương pháp | Generation (sinh token) | Scoring (chấm điểm) |
| Tốc độ | Chậm (token-by-token) | Nhanh (9× faster) |
| Hallucination | Có thể có | Không |
| Arguments | Có thể sinh sai | Không sinh (chỉ chọn tool) |

### 1.2 Benchmark Results

| Benchmark | CLM | JEV (baseline) |
|-----------|-----|----------------|
| Zero-shot (BFCL v4) | On par | Baseline |
| Terminal-Bench 2.1 | **87.6%** (SOTA) | Failed |
| DeepSWE | **81.6%** (SOTA) | Failed |
| Latency | 9× faster | Baseline |

### 1.3 Ưu điểm của CLM

✅ **Nhanh**: 9× faster latency so với generation models
✅ **Chính xác**: SOTA trên nhiều benchmarks
✅ **Không hallucinate**: Chỉ chọn từ candidates
✅ **Fine-tuning rẻ**: Chỉ 20M trainable parameters
✅ **Caching**: State/action embeddings có thể reuse

---

## 2. Cài đặt và Khởi tạo

In [ ]:
# Install dependencies
!pip uninstall -y torchaudio
!pip install contrastive-lm vllm transformers accelerate bitsandbytes -q
!pip install matplotlib seaborn pandas numpy scipy -q


In [ ]:
import os
import json
import time
import subprocess
import urllib.request
from datetime import datetime
from typing import List, Dict, Any, Optional
from collections import Counter

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Configuration
VLLM_PORT = 8090
CLM_PORT = 8700
VLLM_URL = f"http://localhost:{VLLM_PORT}"
CLM_URL = f"http://localhost:{CLM_PORT}"

# Use 4-bit AWQ quantized model to fit smoothly within Colab Free T4 (15GB VRAM)
VLLM_MODEL = "Qwen/Qwen3-8B-AWQ"

print(f"[{datetime.now().strftime('%H:%M:%S')}] Configuration loaded!")
print(f"Embedding Model: {VLLM_MODEL}")
print(f"vLLM Port: {VLLM_PORT} | CLM Port: {CLM_PORT}")


### 2.1 Start vLLM Server

In [ ]:
def start_services(max_retries=60):
    """Start vLLM backend and CLM Server with Tesla T4 optimizations"""
    # Kill existing processes on ports
    subprocess.run("pkill -9 -f vllm || true", shell=True)
    subprocess.run("pkill -9 -f clm-serve || true", shell=True)
    subprocess.run(f"fuser -k {VLLM_PORT}/tcp || true", shell=True)
    subprocess.run(f"fuser -k {CLM_PORT}/tcp || true", shell=True)
    time.sleep(2)
    
    env = os.environ.copy()
    # Force PyTorch SDPA (Tesla T4 does not support FlashAttention-2)
    env["VLLM_ATTENTION_BACKEND"] = "TORCH_SDPA"
    
    # 1. Start vLLM Server
    vllm_cmd = [
        "vllm", "serve", VLLM_MODEL,
        "--served-model-name", "qwen3-8b",
        "--runner", "pooling",
        "--max-model-len", "2048",
        "--port", str(VLLM_PORT),
        "--tensor-parallel-size", "1",
        "--dtype", "float16",
        "--quantization", "awq",  # Explicit standard AWQ for Turing sm_75 (avoids awq_marlin which requires sm_80+)
        "--enforce-eager",
        "--gpu-memory-utilization", "0.80"  # Safer threshold for Colab T4 15GB VRAM
    ]
    
    print("=" * 65)
    print("🚀 Step 1/2: Starting vLLM Server (Embedding Backend)...")
    print(f"Command: {' '.join(vllm_cmd)}")
    print("=" * 65)
    
    # Log to file to avoid 64KB OS pipe buffer deadlock
    vllm_log = open("vllm.log", "w")
    vllm_proc = subprocess.Popen(vllm_cmd, stdout=vllm_log, stderr=subprocess.STDOUT, env=env)
    
    print("\n⏳ Waiting for vLLM to download weights and initialize...")
    vllm_ready = False
    for i in range(max_retries):
        time.sleep(5)
        if vllm_proc.poll() is not None:
            print(f"\n❌ vLLM process crashed with return code {vllm_proc.returncode}!")
            print("--- Detailed crash log (last 80 lines of vllm.log) ---")
            with open("vllm.log", "r") as f:
                lines = f.readlines()
                print("".join(lines[-80:] if len(lines) >= 80 else lines))
            return False
        
        try:
            req = urllib.request.urlopen(f"{VLLM_URL}/v1/models", timeout=2)
            if req.status == 200:
                print(f"\n✅ vLLM Server is READY at {VLLM_URL}!")
                vllm_ready = True
                break
        except Exception:
            pass
        
        # Print heartbeat progress
        if (i + 1) % 2 == 0:
            try:
                with open("vllm.log", "r") as f:
                    last_lines = [l.strip() for l in f.readlines() if l.strip()]
                    last_log = last_lines[-1] if last_lines else ""
                    print(f"  [{(i+1)*5}s / {max_retries*5}s] Latest log: {last_log[:90]}")
            except Exception:
                pass
    
    if not vllm_ready:
        print("\n❌ vLLM server timed out!")
        with open("vllm.log", "r") as f:
            lines = f.readlines()
            print("".join(lines[-80:] if len(lines) >= 80 else lines))
        return False
    
    # 2. Start CLM Server (System One API)
    clm_cmd = [
        "clm-serve",
        "--port", str(CLM_PORT),
        "--emb-url", f"{VLLM_URL}/v1/embeddings",
        "--no-ui"
    ]
    
    print("\n" + "=" * 65)
    print("🚀 Step 2/2: Starting CLM Server (clm-serve)... ")
    print(f"Command: {' '.join(clm_cmd)}")
    print("=" * 65)
    
    clm_log = open("clm_serve.log", "w")
    clm_proc = subprocess.Popen(clm_cmd, stdout=clm_log, stderr=subprocess.STDOUT)
    
    clm_ready = False
    for i in range(30):
        if clm_proc.poll() is not None:
            print(f"\n❌ clm-serve crashed with code {clm_proc.returncode}!")
            with open("clm_serve.log", "r") as f:
                print("".join(f.readlines()[-30:]))
            return False
        try:
            req = urllib.request.urlopen(f"{CLM_URL}/health", timeout=2)
            if req.status == 200:
                print(f"\n✅ CLM Server is READY at {CLM_URL}!")
                clm_ready = True
                break
        except Exception:
            try:
                req = urllib.request.urlopen(f"{CLM_URL}/", timeout=2)
                print(f"\n✅ CLM Server is READY at {CLM_URL}!")
                clm_ready = True
                break
            except Exception:
                pass
        time.sleep(1)
    
    if not clm_ready:
        print("\n⚠️ Check clm_serve.log output:")
        with open("clm_serve.log", "r") as f:
            print("".join(f.readlines()[-20:]))
    
    return vllm_ready and clm_ready

# Start both services
start_services()

### 2.2 Initialize CLM Client

In [ ]:
from clm import CLMClient, Choice, Noul, Score

# Initialize CLM client connected to our local clm-serve instance
try:
    client = CLMClient(base_url=CLM_URL)
except TypeError:
    client = CLMClient()

print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ CLM Client initialized!")
print(f"Target CLM Server: {CLM_URL}")
print(f"Client type: {type(client)}")


---

## 3. Tải Dataset

In [ ]:
import glob
# Mount Google Drive if in Colab environment
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

# Candidate dataset paths
CANDIDATE_PATHS = [
    "/content/drive/MyDrive/vinfast/data/batches/",
    "./eval/tool_call/datasets/multi_turn/batches/",
    "/home/admin123/Documents/fixed-uuid/eval/tool_call/datasets/multi_turn/batches/",
    "/content/batches/"
]

DATASET_PATH = None
for path in CANDIDATE_PATHS:
    if os.path.exists(path) and len(glob.glob(os.path.join(path, "*.json"))) > 0:
        DATASET_PATH = path
        break

if DATASET_PATH is None:
    DATASET_PATH = "/content/drive/MyDrive/vinfast/data/batches/"
    print(f"⚠️ Warning: Using default dataset path: {DATASET_PATH}")
else:
    print(f"✅ Selected dataset path: {DATASET_PATH}")

def load_dataset(dataset_path: str) -> List[Dict]:
    """Load samples from JSON dataset files"""
    all_samples = []
    json_files = sorted(glob.glob(os.path.join(dataset_path, "*.json")))
    
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Found {len(json_files)} JSON files")
    
    for file_path in json_files:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Loading: {os.path.basename(file_path)}")
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                samples = json.load(f)
                all_samples.extend(samples)
                print(f"         └─ Loaded {len(samples)} samples")
        except Exception as e:
            print(f"         └─ Error loading {file_path}: {e}")
    
    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] 📊 Total samples: {len(all_samples)}")
    return all_samples

# Load dataset
samples = load_dataset(DATASET_PATH)


### 3.1 Dataset Statistics

In [ ]:
# Analyze dataset
def analyze_dataset(samples: List[Dict]) -> Dict:
    """Analyze dataset statistics"""
    stats = {
        'total_samples': len(samples),
        'total_conversations': len(set(s.get('conversation_id', i) for i, s in enumerate(samples))),
        'tools_distribution': Counter(),
        'turn_counts': [],
        'topics': Counter(),
    }
    
    for sample in samples:
        for tool in sample.get('tools', []):
            fname = tool.get('function', {}).get('name', 'unknown')
            stats['tools_distribution'][fname] += 1
        
        stats['turn_counts'].append(sample.get('turn_count', len(sample.get('messages', []))))
        topic = sample.get('conversation_topic', 'Unknown')
        stats['topics'][topic] += 1
    
    return stats

# Analyze
dataset_stats = analyze_dataset(samples)

print("=" * 60)
print("📊 DATASET STATISTICS")
print("=" * 60)
print(f"Total samples: {dataset_stats['total_samples']}")
print(f"Unique conversations: {dataset_stats['total_conversations']}")
if dataset_stats['turn_counts']:
    print(f"\n📈 Turn counts: min={min(dataset_stats['turn_counts'])}, max={max(dataset_stats['turn_counts'])}, avg={np.mean(dataset_stats['turn_counts']):.1f}")

print(f"\n🔧 Tools distribution:")
for tool, count in dataset_stats['tools_distribution'].most_common():
    pct = (count / dataset_stats['total_samples'] * 100) if dataset_stats['total_samples'] > 0 else 0
    print(f"   • {tool}: {count} ({pct:.1f}%)")


---

## 4. Inference Engine

In [ ]:
def extract_ground_truth_all_turns(sample: Dict) -> List[Dict]:
    """
    Extract ground truth tool call cho TẤT CẢ turns trong conversation.
    
    Mỗi conversation có nhiều turns, mỗi turn có thể có tool call.
    
    Args:
        sample: Sample dict with 'messages'
    
    Returns:
        List of dicts với 'turn_index', 'tool_name', 'arguments', 'messages_so_far'
    """
    messages = sample.get('messages', [])
    ground_truth_turns = []
    
    turn_index = 0
    conversation_history = []
    
    for msg in messages:
        role = msg.get('role')
        if role in ('system', 'user', 'tool'):
            conversation_history.append(msg)
        elif role == 'assistant':
            if msg.get('tool_calls'):
                tool_call = msg['tool_calls'][0]
                ground_truth_turns.append({
                    'turn_index': turn_index,
                    'tool_name': tool_call.get('function', {}).get('name', ''),
                    'arguments': tool_call.get('function', {}).get('arguments', '{}'),
                    'messages_so_far': [m for m in conversation_history]  # Copy history up to this turn
                })
                turn_index += 1
            # Luôn thêm assistant message vào history (dù có content hay tool_calls)
            conversation_history.append(msg)
    
    return ground_truth_turns


def predict_tool_with_clm_at_turn(
    client: CLMClient,
    messages_so_far: List[Dict],
    tools: List[Dict],
    max_turns: Optional[int] = None,
    include_distractors: bool = True
) -> Dict[str, Any]:
    """
    Predict tool call tại một turn cụ thể trong conversation.
    
    Args:
        client: CLMClient instance
        messages_so_far: Messages cho đến turn hiện tại
        tools: Available tools
        max_turns: Số turns ngữ cảnh đưa vào state (None = lấy tất cả)
        include_distractors: Thêm distractor tools nếu chỉ có 1 tool candidate
    
    Returns:
        Dict với predicted_tool, probability, probabilities, etc.
    """
    # Format conversation as state (is_turn_history=True để giữ đầy đủ ngữ cảnh các lượt trước)
    state = format_conversation(messages_so_far, max_turns=max_turns, is_turn_history=True)
    
    # Convert tools to criteria (candidates)
    criteria = extract_tools_as_criteria(tools, include_distractors=include_distractors)
    
    if not criteria:
        return {
            'error': 'No tools available',
            'predicted_tool': None,
            'all_criteria': []
        }
    
    try:
        # Run CLM inference
        result = client.system_one(
            state=state,
            questions={
                'tool': Choice(
                    instructions="Select the most appropriate tool to call based on the conversation context. Consider what information the user needs and which tool would best provide that information.",
                    criteria=criteria
                )
            }
        )
        
        # Safely extract from SystemOneResponse answers dictionary
        tool_ans = result.answers.get("tool") if hasattr(result, "answers") else None
        if tool_ans is not None:
            predicted = getattr(tool_ans, "choice", None)
            prob = getattr(tool_ans, "confidence", 0.0)
            probs = getattr(tool_ans, "probabilities", {})
        else:
            predicted = getattr(result, "tool", None)
            prob = getattr(result, "tool_prob", 0.0)
            probs = {}
            
        return {
            'predicted_tool': predicted,
            'probability': prob,
            'probabilities': probs,
            'all_criteria': list(criteria.keys()),
            'num_candidates': len(criteria),
            'state_length': len(state)
        }
        
    except Exception as e:
        return {
            'error': str(e),
            'predicted_tool': None,
            'all_criteria': list(criteria.keys()) if criteria else []
        }


print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ Turn-by-turn evaluation functions defined!")

In [ ]:
# Candidate distractor tools to ensure CLM performs meaningful choice ranking
GLOBAL_DISTRACTOR_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Tra cứu thông tin dự báo thời tiết, nhiệt độ, lượng mưa, độ ẩm các tỉnh thành Việt Nam.",
            "parameters": {"type": "object", "properties": {"keyword": {"type": "string"}}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_gold_price",
            "description": "Tra cứu giá vàng thị trường thời gian thực hoặc xu hướng lịch sử giá vàng tại Việt Nam (SJC, PNJ, DOJI).",
            "parameters": {"type": "object", "properties": {"action": {"type": "string"}}}
        }
    }
]

def format_conversation(messages: List[Dict], max_turns: Optional[int] = 3, is_turn_history: bool = False) -> str:
    """
    Format messages into a conversation string for CLM state.
    - If is_turn_history is False: cuts off right at the user prompt immediately preceding the target assistant tool call.
    - If is_turn_history is True: messages already end at the target user prompt, so prior turns are preserved as dialogue context.
    """
    if is_turn_history:
        relevant_messages = messages
    else:
        # 1. Locate the target tool call turn
        target_idx = len(messages)
        for idx in range(len(messages) - 1, -1, -1):
            if messages[idx].get("role") == "assistant" and messages[idx].get("tool_calls"):
                target_idx = idx
                break
        relevant_messages = messages[:target_idx]
        
    relevant_messages = [m for m in relevant_messages if m.get('role') != 'system']
    if max_turns:
        relevant_messages = relevant_messages[-max_turns*2:]
    
    formatted_parts = []
    for msg in relevant_messages:
        role = msg.get('role', 'user').capitalize()
        content = msg.get('content') or ''
        
        if msg.get('tool_calls'):
            for tc in msg['tool_calls']:
                fname = tc.get('function', {}).get('name', 'tool')
                formatted_parts.append(f"Assistant calls tool: {fname}")
        elif msg.get('tool_call_id') or role == 'Tool':
            content_preview = content[:100]
            if content_preview:
                formatted_parts.append(f"Tool result: {content_preview}...")
        elif content:
            if len(content) > 500:
                content = content[:500] + "..."
            formatted_parts.append(f"{role}: {content}")
    
    return "\n".join(formatted_parts)


def extract_tools_as_criteria(tools: List[Dict], include_distractors: bool = True) -> Dict[str, str]:
    """
    Convert tools to CLM Choice criteria format.
    Includes distractor tools if single candidate is provided.
    """
    all_tools = list(tools)
    if include_distractors and len(all_tools) == 1:
        existing_names = {t.get("function", {}).get("name") for t in all_tools}
        for dt in GLOBAL_DISTRACTOR_TOOLS:
            if dt["function"]["name"] not in existing_names:
                all_tools.append(dt)
    
    criteria = {}
    for tool in all_tools:
        func = tool.get('function', tool)
        name = func.get('name', '')
        desc = func.get('description', '')
        
        params = func.get('parameters', {})
        if 'properties' in params:
            param_names = list(params['properties'].keys())
            if param_names:
                desc += f" (parameters: {', '.join(param_names)})"
        
        criteria[name] = desc
    
    return criteria


def extract_ground_truth(sample: Dict) -> Optional[Dict]:
    """Extract ground truth tool call from sample"""
    messages = sample.get('messages', [])
    for msg in reversed(messages):
        if msg.get('role') == 'assistant' and msg.get('tool_calls'):
            tool_call = msg['tool_calls'][0]
            return {
                'tool_name': tool_call['function']['name'],
                'arguments': tool_call['function'].get('arguments', '{}')
            }
    return None

print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ Helper functions defined!")

### 4.2 CLM Inference Function

In [ ]:
def predict_tool_with_clm(
    client: CLMClient,
    messages: List[Dict],
    tools: List[Dict],
    max_turns: int = 3,
    include_distractors: bool = True
) -> Dict[str, Any]:
    """
    Predict tool call using CLM with proper SystemOneResponse parsing.
    """
    state = format_conversation(messages, max_turns)
    criteria = extract_tools_as_criteria(tools, include_distractors=include_distractors)
    
    if not criteria:
        return {
            'error': 'No tools available',
            'predicted_tool': None,
            'all_criteria': []
        }
    
    try:
        result = client.system_one(
            state=state,
            questions={
                'tool': Choice(
                    instructions="Select the most appropriate tool to call based on the conversation context. Consider what information the user needs and which tool would best provide that information.",
                    criteria=criteria
                )
            }
        )
        
        # Safely extract from SystemOneResponse answers dictionary
        tool_ans = result.answers.get("tool") if hasattr(result, "answers") else None
        if tool_ans is not None:
            predicted = getattr(tool_ans, "choice", None)
            prob = getattr(tool_ans, "confidence", 0.0)
            probs = getattr(tool_ans, "probabilities", {})
        else:
            predicted = getattr(result, "tool", None)
            prob = getattr(result, "tool_prob", 0.0)
            probs = {}
        
        return {
            'predicted_tool': predicted,
            'probability': prob,
            'probabilities': probs,
            'all_criteria': list(criteria.keys()),
            'num_candidates': len(criteria),
            'state_length': len(state)
        }
        
    except Exception as e:
        return {
            'error': str(e),
            'predicted_tool': None,
            'all_criteria': list(criteria.keys()) if criteria else []
        }

print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ CLM inference function defined!")


---

## 5. Đánh giá toàn diện

In [ ]:
def comprehensive_evaluate(
    client: CLMClient,
    samples: List[Dict],
    max_samples: int = 100,
    max_turns: int = 3,
    include_distractors: bool = True,
    verbose: bool = True
) -> Dict[str, Any]:
    """
    Comprehensive evaluation of CLM on function calling task.
    """
    results = {
        'total': 0,
        'correct': 0,
        'incorrect': 0,
        'errors': 0,
        'latencies': [],
        'probabilities': [],
        'details': [],
        'prediction_distribution': Counter(),
        'actual_distribution': Counter(),
        'confidence_bins': {'high': 0, 'medium': 0, 'low': 0},
        'correct_by_confidence': {'high': {'correct': 0, 'total': 0},
                                   'medium': {'correct': 0, 'total': 0},
                                   'low': {'correct': 0, 'total': 0}}
    }
    
    samples_to_eval = min(max_samples, len(samples))
    
    if verbose:
        print(f"\n[{datetime.now().strftime('%H:%M:%S')}] 🔬 Starting evaluation ({samples_to_eval} samples)")
        print("=" * 60)
    
    for i, sample in enumerate(samples[:samples_to_eval]):
        ground_truth = extract_ground_truth(sample)
        
        if not ground_truth:
            results['errors'] += 1
            results['details'].append({
                'sample_id': sample.get('conversation_id', f'sample_{i}'),
                'error': 'No ground truth'
            })
            continue
        
        start_time = time.time()
        prediction = predict_tool_with_clm(
            client, 
            sample['messages'], 
            sample.get('tools', []),
            max_turns=max_turns,
            include_distractors=include_distractors
        )
        latency = time.time() - start_time
        
        results['latencies'].append(latency)
        results['total'] += 1
        results['actual_distribution'][ground_truth['tool_name']] += 1
        
        if 'error' in prediction and prediction['error']:
            results['errors'] += 1
            results['details'].append({
                'sample_id': sample.get('conversation_id', f'sample_{i}'),
                'error': prediction['error'],
                'correct': False
            })
        else:
            predicted = prediction.get('predicted_tool')
            actual = ground_truth['tool_name']
            probability = prediction.get('probability', 0)
            is_correct = (predicted == actual)
            
            results['prediction_distribution'][predicted] += 1
            results['probabilities'].append(probability)
            
            if probability >= 0.9:
                results['confidence_bins']['high'] += 1
                results['correct_by_confidence']['high']['total'] += 1
                if is_correct:
                    results['correct_by_confidence']['high']['correct'] += 1
            elif probability >= 0.7:
                results['confidence_bins']['medium'] += 1
                results['correct_by_confidence']['medium']['total'] += 1
                if is_correct:
                    results['correct_by_confidence']['medium']['correct'] += 1
            else:
                results['confidence_bins']['low'] += 1
                results['correct_by_confidence']['low']['total'] += 1
                if is_correct:
                    results['correct_by_confidence']['low']['correct'] += 1
            
            if is_correct:
                results['correct'] += 1
            else:
                results['incorrect'] += 1
            
            results['details'].append({
                'sample_id': sample.get('conversation_id', f'sample_{i}'),
                'topic': sample.get('conversation_topic', 'Unknown'),
                'predicted': predicted,
                'actual': actual,
                'correct': is_correct,
                'probability': probability,
                'latency': latency,
                'num_tools': len(sample.get('tools', []))
            })
        
        if verbose and (i + 1) % 10 == 0:
            acc_now = (results['correct'] / results['total']) * 100 if results['total'] > 0 else 0
            avg_lat = np.mean(results['latencies']) if results['latencies'] else 0
            print(f"[{datetime.now().strftime('%H:%M:%S')}] Progress: {i+1}/{samples_to_eval} -> Accuracy: {acc_now:.1f}% (Avg Latency: {avg_lat*1000:.1f}ms)")
    
    # Calculate overall metrics
    results['accuracy'] = results['correct'] / results['total'] if results['total'] > 0 else 0
    total_processed = results['total'] + results['errors']
    results['error_rate'] = results['errors'] / total_processed if total_processed > 0 else 0
    results['avg_latency'] = np.mean(results['latencies']) if results['latencies'] else 0
    results['median_latency'] = np.median(results['latencies']) if results['latencies'] else 0
    results['p95_latency'] = np.percentile(results['latencies'], 95) if results['latencies'] else 0
    results['avg_probability'] = np.mean(results['probabilities']) if results['probabilities'] else 0
    
    for level in results['correct_by_confidence']:
        data = results['correct_by_confidence'][level]
        data['accuracy'] = data['correct'] / data['total'] if data['total'] > 0 else 0
    
    if verbose:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ Single-turn evaluation defined!")
    
    return results


def comprehensive_evaluate_all_turns(
    client: CLMClient,
    samples: List[Dict],
    max_samples: int = 64,
    max_turns_context: Optional[int] = None,
    include_distractors: bool = True,
    verbose: bool = True
) -> Dict[str, Any]:
    """
    Comprehensive evaluation of CLM on ALL turns of each conversation in dataset.
    
    Args:
        client: CLMClient instance
        samples: List of dataset samples
        max_samples: Maximum number of conversations to evaluate
        max_turns_context: Number of context turns to include in state (None = full history)
        include_distractors: Add distractor tools if single candidate
        verbose: Print evaluation progress
        
    Returns:
        Dict with comprehensive turn-by-turn evaluation metrics
    """
    results = {
        'total_conversations': 0,
        'total_turns': 0,
        'correct': 0,
        'incorrect': 0,
        'errors': 0,
        'accuracy': 0.0,
        'avg_latency': 0.0,
        'avg_probability': 0.0,
        'latencies': [],
        'probabilities': [],
        'prediction_distribution': Counter(),
        'actual_distribution': Counter(),
        'by_turn': {},
        'conversations': []
    }
    
    samples_to_eval = samples[:max_samples]
    total_convs = len(samples_to_eval)
    
    if verbose:
        print(f"\n[{datetime.now().strftime('%H:%M:%S')}] 🔬 Starting Turn-by-Turn Evaluation ({total_convs} conversations)")
        print("=" * 70)
        
    for conv_idx, sample in enumerate(samples_to_eval):
        conv_id = sample.get('conversation_id', f'conv_{conv_idx+1}')
        topic = sample.get('conversation_topic', 'Unknown')
        tools = sample.get('tools', [])
        
        gt_turns = extract_ground_truth_all_turns(sample)
        
        if not gt_turns:
            results['errors'] += 1
            results['conversations'].append({
                'conversation_id': conv_id,
                'topic': topic,
                'total_turns': 0,
                'correct_turns': 0,
                'incorrect_turns': 0,
                'errors': 1,
                'turns': []
            })
            if verbose:
                print(f"[{datetime.now().strftime('%H:%M:%S')}] ⚠️ Conv {conv_idx+1}/{total_convs} ({conv_id}): No tool calls found!")
            continue
            
        conv_turn_results = []
        for turn in gt_turns:
            turn_idx = turn['turn_index']
            expected_tool = turn['tool_name']
            messages_so_far = turn['messages_so_far']
            
            start_time = time.time()
            prediction = predict_tool_with_clm_at_turn(
                client=client,
                messages_so_far=messages_so_far,
                tools=tools,
                max_turns=max_turns_context,
                include_distractors=include_distractors
            )
            latency = time.time() - start_time
            
            results['latencies'].append(latency)
            results['total_turns'] += 1
            results['actual_distribution'][expected_tool] += 1
            
            if turn_idx not in results['by_turn']:
                results['by_turn'][turn_idx] = {
                    'total': 0,
                    'correct': 0,
                    'incorrect': 0,
                    'errors': 0,
                    'latencies': [],
                    'probabilities': [],
                    'accuracy': 0.0,
                    'avg_latency': 0.0,
                    'avg_probability': 0.0
                }
            t_stats = results['by_turn'][turn_idx]
            t_stats['total'] += 1
            t_stats['latencies'].append(latency)
            
            if prediction.get('error'):
                results['errors'] += 1
                t_stats['errors'] += 1
                t_stats['probabilities'].append(0.0)
                results['probabilities'].append(0.0)
                conv_turn_results.append({
                    'turn_index': turn_idx,
                    'predicted': None,
                    'actual': expected_tool,
                    'correct': False,
                    'probability': 0.0,
                    'latency': latency,
                    'error': prediction['error']
                })
            else:
                predicted_tool = prediction.get('predicted_tool')
                prob = prediction.get('probability', 0.0)
                is_correct = (predicted_tool == expected_tool)
                
                results['probabilities'].append(prob)
                t_stats['probabilities'].append(prob)
                if predicted_tool:
                    results['prediction_distribution'][predicted_tool] += 1
                
                if is_correct:
                    results['correct'] += 1
                    t_stats['correct'] += 1
                else:
                    results['incorrect'] += 1
                    t_stats['incorrect'] += 1
                    
                conv_turn_results.append({
                    'turn_index': turn_idx,
                    'predicted': predicted_tool,
                    'actual': expected_tool,
                    'correct': is_correct,
                    'probability': prob,
                    'latency': latency,
                    'error': None
                })
        
        correct_count = sum(1 for t in conv_turn_results if t['correct'])
        inc_count = sum(1 for t in conv_turn_results if not t['correct'] and not t.get('error'))
        err_count = sum(1 for t in conv_turn_results if t.get('error'))
        
        results['conversations'].append({
            'conversation_id': conv_id,
            'topic': topic,
            'total_turns': len(conv_turn_results),
            'correct_turns': correct_count,
            'incorrect_turns': inc_count,
            'errors': err_count,
            'turns': conv_turn_results
        })
        
        if verbose and ((conv_idx + 1) % 5 == 0 or (conv_idx + 1) == total_convs):
            acc_now = (results['correct'] / results['total_turns'] * 100) if results['total_turns'] > 0 else 0
            avg_lat = (np.mean(results['latencies']) * 1000) if results['latencies'] else 0
            print(f"[{datetime.now().strftime('%H:%M:%S')}] Progress: {conv_idx+1}/{total_convs} convs | Total turns: {results['total_turns']} | Accuracy: {acc_now:.1f}% | Avg Latency: {avg_lat:.1f}ms")
            
    # Calculate aggregated metrics
    results['total_conversations'] = len(results['conversations'])
    results['accuracy'] = (results['correct'] / results['total_turns']) if results['total_turns'] > 0 else 0.0
    results['avg_latency'] = float(np.mean(results['latencies'])) if results['latencies'] else 0.0
    results['avg_probability'] = float(np.mean(results['probabilities'])) if results['probabilities'] else 0.0
    
    for turn_idx, t_stats in results['by_turn'].items():
        t_stats['accuracy'] = (t_stats['correct'] / t_stats['total']) if t_stats['total'] > 0 else 0.0
        t_stats['avg_latency'] = float(np.mean(t_stats['latencies'])) if t_stats['latencies'] else 0.0
        t_stats['avg_probability'] = float(np.mean(t_stats['probabilities'])) if t_stats['probabilities'] else 0.0
        
    if verbose:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ Turn-by-Turn Evaluation complete!")
        
    return results

print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ Both single-turn and turn-by-turn evaluation functions defined!")

### 5.1 Run Evaluation

In [ ]:
# Run comprehensive evaluation
print("=" * 70)
print("🔬 COMPREHENSIVE CLM EVALUATION")
print("=" * 70)

MAX_SAMPLES = 100  # Adjust based on dataset size
MAX_TURNS = 3

start_total = time.time()
results = comprehensive_evaluate(
    client=client,
    samples=samples,
    max_samples=MAX_SAMPLES,
    max_turns=MAX_TURNS,
    verbose=True
)
total_time = time.time() - start_total

print(f"\n⏱️ Total evaluation time: {total_time:.1f}s")

### 5.2 Run Turn-by-Turn Evaluation

Đánh giá **tất cả 5 turns** trong mỗi conversation:


In [ ]:
# ============================================================
# TURN-BY-TURN EVALUATION
# ============================================================
# Đánh giá TẤT CẢ các turns trong mỗi conversation

print("=" * 70)
print("🔬 TURN-BY-TURN COMPREHENSIVE EVALUATION")
print("=" * 70)
print("Đánh giá tất cả turns trong mỗi conversation...")
print("=" * 70)

MAX_CONVERSATIONS = 64  # Tất cả 64 conversations

start_total = time.time()
results_all_turns = comprehensive_evaluate_all_turns(
    client=client,
    samples=samples,
    max_samples=MAX_CONVERSATIONS,
    verbose=True
)
total_time = time.time() - start_total

print(f"\n⏱️ Total evaluation time: {total_time:.1f}s")
print(f"📊 Total conversations: {results_all_turns['total_conversations']}")
print(f"📊 Total turns: {results_all_turns['total_turns']}")
print(f"📊 Correct: {results_all_turns['correct']}")
print(f"📊 Incorrect: {results_all_turns['incorrect']}")
print(f"🎯 Accuracy: {results_all_turns['accuracy']*100:.2f}%")

### 5.3 Turn-by-Turn Results Summary

In [ ]:
# ============================================================
# TURN-BY-TURN RESULTS SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("📊 TURN-BY-TURN RESULTS SUMMARY")
print("=" * 70)

print(f"""
┌─────────────────────────────────────────────────────────────────────┐
│                    OVERALL METRICS                                     │
├─────────────────────────────────────────────────────────────────────┤
│  Total conversations:      {results_all_turns['total_conversations']:>10}                          │
│  Total turns evaluated:    {results_all_turns['total_turns']:>10}                          │
│  Correct predictions:      {results_all_turns['correct']:>10}                          │
│  Incorrect predictions:    {results_all_turns['incorrect']:>10}                          │
│  Errors:                  {results_all_turns['errors']:>10}                          │
├─────────────────────────────────────────────────────────────────────┤
│  🎯 OVERALL ACCURACY:     {results_all_turns['accuracy']*100:>10.2f}%                          │
├─────────────────────────────────────────────────────────────────────┤
│                    LATENCY METRICS                                   │
├─────────────────────────────────────────────────────────────────────┤
│  Average latency:         {results_all_turns['avg_latency']*1000:>10.2f} ms                         │
│  Average probability:     {results_all_turns['avg_probability']:>10.4f}                         │
└─────────────────────────────────────────────────────────────────────┘
""")

# Accuracy by Turn Position
print("\n📈 Accuracy by Turn Position:")
print("-" * 70)
print(f"{'Turn':<10} {'Total':<10} {'Correct':<10} {'Incorrect':<12} {'Accuracy':<12} {'Avg Prob':<10}")
print("-" * 70)

for turn_idx in sorted(results_all_turns['by_turn'].keys()):
    turn_stats = results_all_turns['by_turn'][turn_idx]
    valid = turn_stats['total'] - turn_stats['errors']
    accuracy = turn_stats['accuracy'] * 100 if valid > 0 else 0
    avg_prob = turn_stats.get('avg_probability', 0)
    print(f"Turn {turn_idx + 1:<4} {turn_stats['total']:<10} {turn_stats['correct']:<10} {turn_stats['incorrect']:<12} {accuracy:.2f}%{'':<6} {avg_prob:.4f}")

print("-" * 70)

# Per-Conversation Stats
print("\n📊 Per-Conversation Summary (First 10 conversations):")
print("-" * 70)

for i, conv in enumerate(results_all_turns['conversations'][:10]):
    acc = (conv['correct_turns'] / conv['total_turns'] * 100) if conv['total_turns'] > 0 else 0
    print(f"  {i+1}. {conv['conversation_id']}: {conv['correct_turns']}/{conv['total_turns']} correct ({acc:.0f}%) - {conv['topic'][:50]}...")

### 6. Visualize Turn-by-Turn Results

In [ ]:
# ============================================================
# TURN-BY-TURN VISUALIZATION
# ============================================================

fig = plt.figure(figsize=(18, 14))

# 1. Accuracy by Turn Position (Bar Chart)
ax1 = fig.add_subplot(2, 3, 1)
turn_indices = sorted(results_all_turns['by_turn'].keys())
accuracies_by_turn = [results_all_turns['by_turn'][t]['accuracy'] * 100 for t in turn_indices]
counts_by_turn = [results_all_turns['by_turn'][t]['total'] for t in turn_indices]

x_pos = np.arange(len(turn_indices))
bars = ax1.bar(x_pos, accuracies_by_turn, color='#3498db', alpha=0.8, edgecolor='black')

# Add count labels on bars
for bar, count in zip(bars, counts_by_turn):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
             f'n={count}', ha='center', va='bottom', fontsize=9)

ax1.set_xlabel('Turn Position', fontsize=10)
ax1.set_ylabel('Accuracy (%)', fontsize=10)
ax1.set_title('Accuracy by Turn Position', fontsize=12, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels([f'Turn {t+1}' for t in turn_indices])
ax1.set_ylim(0, 110)
ax1.axhline(100, color='green', linestyle='--', alpha=0.5, label='100%')
ax1.grid(True, alpha=0.3, axis='y')

# 2. Latency by Turn
ax2 = fig.add_subplot(2, 3, 2)
latencies_by_turn = [results_all_turns['by_turn'][t].get('avg_latency', 0) * 1000 for t in turn_indices]
ax2.bar(x_pos, latencies_by_turn, color='#e74c3c', alpha=0.8, edgecolor='black')
ax2.set_xlabel('Turn Position', fontsize=10)
ax2.set_ylabel('Avg Latency (ms)', fontsize=10)
ax2.set_title('Average Latency by Turn Position', fontsize=12, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels([f'Turn {t+1}' for t in turn_indices])
ax2.grid(True, alpha=0.3, axis='y')

# 3. Probability by Turn
ax3 = fig.add_subplot(2, 3, 3)
probs_by_turn = [results_all_turns['by_turn'][t].get('avg_probability', 0) for t in turn_indices]
ax3.bar(x_pos, probs_by_turn, color='#9b59b6', alpha=0.8, edgecolor='black')
ax3.set_xlabel('Turn Position', fontsize=10)
ax3.set_ylabel('Avg Probability', fontsize=10)
ax3.set_title('Average Probability by Turn Position', fontsize=12, fontweight='bold')
ax3.set_xticks(x_pos)
ax3.set_xticklabels([f'Turn {t+1}' for t in turn_indices])
ax3.set_ylim(0, 1.1)
ax3.axhline(0.9, color='red', linestyle='--', alpha=0.5, label='High threshold')
ax3.axhline(0.7, color='orange', linestyle='--', alpha=0.5, label='Medium threshold')
ax3.grid(True, alpha=0.3, axis='y')

# 4. Overall Accuracy Pie Chart
ax4 = fig.add_subplot(2, 3, 4)
sizes = [results_all_turns['correct'], results_all_turns['incorrect']]
labels = [f'Correct\n({results_all_turns["correct"]})', f'Incorrect\n({results_all_turns["incorrect"]})']
colors = ['#2ecc71', '#e74c3c']
if sum(sizes) == 0:
    sizes = [1, 0]
    labels = ['No data', '']
ax4.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax4.set_title(f'Turn-by-Turn Accuracy\n({results_all_turns["accuracy"]*100:.1f}%)', fontsize=12, fontweight='bold')

# 5. Stacked Bar - Correct vs Incorrect by Turn
ax5 = fig.add_subplot(2, 3, 5)
correct_by_turn = [results_all_turns['by_turn'][t]['correct'] for t in turn_indices]
incorrect_by_turn = [results_all_turns['by_turn'][t]['incorrect'] for t in turn_indices]
width = 0.6
ax5.bar(x_pos, correct_by_turn, width, label='Correct', color='#2ecc71')
ax5.bar(x_pos, incorrect_by_turn, width, bottom=correct_by_turn, label='Incorrect', color='#e74c3c')
ax5.set_xlabel('Turn Position', fontsize=10)
ax5.set_ylabel('Count', fontsize=10)
ax5.set_title('Correct vs Incorrect by Turn', fontsize=12, fontweight='bold')
ax5.set_xticks(x_pos)
ax5.set_xticklabels([f'Turn {t+1}' for t in turn_indices])
ax5.legend()
ax5.grid(True, alpha=0.3, axis='y')

# 6. Line Chart - Metrics by Turn
ax6 = fig.add_subplot(2, 3, 6)
ax6.plot(turn_indices, accuracies_by_turn, 'b-o', label='Accuracy (%)', linewidth=2, markersize=8)
ax6.plot(turn_indices, probs_by_turn, 'g-s', label='Avg Probability', linewidth=2, markersize=8)
ax6.set_xlabel('Turn Position', fontsize=10)
ax6.set_ylabel('Score', fontsize=10)
ax6.set_title('Metrics Trend Across Turns', fontsize=12, fontweight='bold')
ax6.set_xticks(turn_indices)
ax6.set_xticklabels([f'Turn {t+1}' for t in turn_indices])
ax6.set_ylim(0, 1.1)
ax6.legend()
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('clm_turn_by_turn_plots.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] 📊 Turn-by-turn plots saved to: clm_turn_by_turn_plots.png")

---

## 8. Xuất báo cáo

In [ ]:
# ============================================================
# EXPORT TURN-BY-TURN RESULTS
# ============================================================

# Create output directory
OUTPUT_BASE = "/content/drive/MyDrive/vinfast/jev_clm/results/clm"
try:
    os.makedirs(OUTPUT_BASE, exist_ok=True)
except Exception:
    OUTPUT_BASE = "./results/clm"
    os.makedirs(OUTPUT_BASE, exist_ok=True)

lat_arr = results_all_turns['latencies'] if results_all_turns['latencies'] else [0.0]
prob_arr = results_all_turns['probabilities'] if results_all_turns['probabilities'] else [0.0]

# Prepare export data
export_data = {
    'model': 'CLM-v0.1-8B',
    'model_type': 'Contrastive Language Model (System One)',
    'base_model': 'Qwen/Qwen3-8B-AWQ',
    'evaluation_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'dataset_path': DATASET_PATH,
    'max_conversations': MAX_CONVERSATIONS,
    
    # Overall Metrics
    'total_conversations': results_all_turns['total_conversations'],
    'total_turns': results_all_turns['total_turns'],
    'correct': results_all_turns['correct'],
    'incorrect': results_all_turns['incorrect'],
    'errors': results_all_turns['errors'],
    'accuracy': results_all_turns['accuracy'],
    'accuracy_percent': results_all_turns['accuracy'] * 100,
    
    # Latency Metrics
    'latency': {
        'mean_ms': results_all_turns['avg_latency'] * 1000,
        'min_ms': np.min(lat_arr) * 1000,
        'max_ms': np.max(lat_arr) * 1000,
        'std_ms': np.std(lat_arr) * 1000
    },
    
    # Probability Metrics
    'probability': {
        'mean': results_all_turns['avg_probability'],
        'std': float(np.std(prob_arr)),
        'min': float(np.min(prob_arr)),
        'max': float(np.max(prob_arr))
    },
    
    # Accuracy by Turn Position
    'accuracy_by_turn': {
        f'turn_{t+1}': {
            'total': results_all_turns['by_turn'][t]['total'],
            'correct': results_all_turns['by_turn'][t]['correct'],
            'incorrect': results_all_turns['by_turn'][t]['incorrect'],
            'accuracy': results_all_turns['by_turn'][t]['accuracy'],
            'avg_probability': results_all_turns['by_turn'][t].get('avg_probability', 0),
            'avg_latency_ms': results_all_turns['by_turn'][t].get('avg_latency', 0) * 1000
        }
        for t in results_all_turns['by_turn'].keys()
    },
    
    # Distribution
    'actual_tool_distribution': dict(results_all_turns['actual_distribution']),
    'predicted_tool_distribution': dict(results_all_turns['prediction_distribution']),
    
    # Per-conversation results (detailed)
    'conversations': results_all_turns['conversations']
}

# Save JSON
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
json_file = f"{OUTPUT_BASE}/clm_turn_by_turn_results_{timestamp}.json"

with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(export_data, f, indent=2, ensure_ascii=False)

print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ Turn-by-turn results saved to: {json_file}")

In [ ]:
# ============================================================
# SAVE TURN-BY-TURN SUMMARY REPORT
# ============================================================

# Create summary report
summary_file = f"{OUTPUT_BASE}/clm_turn_by_turn_summary_{timestamp}.txt"

with open(summary_file, 'w', encoding='utf-8') as f:
    f.write("=" * 70 + "\n")
    f.write("CLM - TURN-BY-TURN EVALUATION SUMMARY REPORT\n")
    f.write("=" * 70 + "\n\n")
    
    f.write("## Model Information\n")
    f.write("-" * 40 + "\n")
    f.write(f"Model:           CLM-v0.1-8B\n")
    f.write(f"Type:            Contrastive Language Model (System One)\n")
    f.write(f"Base Model:      Qwen/Qwen3-8B-AWQ\n")
    f.write(f"Evaluation Date: {export_data['evaluation_date']}\n")
    f.write(f"Dataset Path:    {DATASET_PATH}\n")
    f.write(f"Conversations:   {MAX_CONVERSATIONS}\n\n")
    
    f.write("## Overall Results\n")
    f.write("-" * 40 + "\n")
    f.write(f"Total Conversations: {results_all_turns['total_conversations']}\n")
    f.write(f"Total Turns:         {results_all_turns['total_turns']}\n")
    f.write(f"Correct:             {results_all_turns['correct']}\n")
    f.write(f"Incorrect:           {results_all_turns['incorrect']}\n")
    f.write(f"Accuracy:            {results_all_turns['accuracy']*100:.2f}%\n\n")
    
    f.write("## Accuracy by Turn Position\n")
    f.write("-" * 40 + "\n")
    f.write(f"{'Turn':<10} {'Total':<10} {'Correct':<10} {'Incorrect':<12} {'Accuracy':<12}\n")
    f.write("-" * 54 + "\n")
    
    for turn_idx in sorted(results_all_turns['by_turn'].keys()):
        turn_stats = results_all_turns['by_turn'][turn_idx]
        valid = turn_stats['total'] - turn_stats['errors']
        accuracy = turn_stats['accuracy'] * 100 if valid > 0 else 0
        f.write(f"Turn {turn_idx + 1:<4} {turn_stats['total']:<10} {turn_stats['correct']:<10} {turn_stats['incorrect']:<12} {accuracy:.2f}%\n")
    
    f.write("\n## Per-Conversation Results\n")
    f.write("-" * 40 + "\n")
    for i, conv in enumerate(results_all_turns['conversations']):
        acc = (conv['correct_turns'] / conv['total_turns'] * 100) if conv['total_turns'] > 0 else 0
        f.write(f"{conv['conversation_id']}: {conv['correct_turns']}/{conv['total_turns']} ({acc:.0f}%)\n")

print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ Summary saved to: {summary_file}")

In [ ]:
# ============================================================
# SAVE PLOTS TO DRIVE
# ============================================================

import shutil

# Copy turn-by-turn plots to Drive
plot_file = 'clm_turn_by_turn_plots.png'
plot_dest = f"{OUTPUT_BASE}/clm_turn_by_turn_plots_{timestamp}.png"
if os.path.exists(plot_file):
    shutil.copy(plot_file, plot_dest)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ Plots saved to: {plot_dest}")
else:
    print(f"[{datetime.now().strftime('%H:%M:%S')}] ⚠️ Plot file {plot_file} not found")

print("\n" + "=" * 70)
print("📁 OUTPUT FILES (Turn-by-Turn Evaluation)")
print("=" * 70)
print(f"1. JSON Results:      {json_file}")
print(f"2. Summary Report:    {summary_file}")
print(f"3. Plots:             {plot_dest}")
print(f"\nOutput directory: {OUTPUT_BASE}")

---

## 9. Detailed Sample Analysis

In [ ]:
# Show sample detailed results
print("=" * 80)
print("📋 DETAILED RESULTS (First 20 samples)")
print("=" * 80)

print(f"{'#':<4} {'Sample ID':<25} {'Predicted':<15} {'Actual':<15} {'Prob':<8} {'Status':<10}")
print("-" * 80)

for i, detail in enumerate(results['details'][:20]):
    status = "✅" if detail['correct'] else "❌"
    prob = detail.get('probability', 0)
    print(f"{i+1:<4} {detail['sample_id']:<25} {detail.get('predicted', 'N/A'):<15} {detail.get('actual', 'N/A'):<15} {prob:.4f}   {status}")

In [ ]:
# Show incorrect predictions (if any)
incorrect_samples = [d for d in results['details'] if not d.get('correct', False)]

if incorrect_samples:
    print(f"\n❌ INCORRECT PREDICTIONS ({len(incorrect_samples)} samples)")
    print("=" * 80)
    
    for i, detail in enumerate(incorrect_samples):
        print(f"\n{i+1}. {detail['sample_id']}")
        print(f"   Predicted: {detail.get('predicted', 'N/A')}")
        print(f"   Actual:    {detail.get('actual', 'N/A')}")
        print(f"   Probability: {detail.get('probability', 0):.4f}")
        print(f"   Topic:     {detail.get('topic', 'N/A')}")
else:
    print(f"\n✅ ALL {results['total']} SAMPLES WERE CORRECTLY PREDICTED!")

---

## 10. Conclusion

In [ ]:
from IPython.display import display, Markdown

report_md = f"""## 10. Conclusion

### Key Findings - Turn-by-Turn Evaluation

#### 1. Overall Results
- **Total Conversations**: {results_all_turns['total_conversations']}
- **Total Turns Evaluated**: {results_all_turns['total_turns']}
- **Overall Accuracy**: {results_all_turns['accuracy']*100:.2f}% ({results_all_turns['correct']}/{results_all_turns['total_turns']})

#### 2. Accuracy by Turn Position

| Turn | Total | Correct | Incorrect | Accuracy |
|------|-------|---------|-----------|----------|
"""

for turn_idx in sorted(results_all_turns['by_turn'].keys()):
    turn_stats = results_all_turns['by_turn'][turn_idx]
    valid = turn_stats['total'] - turn_stats['errors']
    accuracy = turn_stats['accuracy'] * 100 if valid > 0 else 0
    report_md += f"| Turn {turn_idx + 1} | {turn_stats['total']} | {turn_stats['correct']} | {turn_stats['incorrect']} | {accuracy:.2f}% |\n"

report_md += f"""
#### 3. Latency
- **Average Latency**: {results_all_turns['avg_latency']*1000:.2f}ms

### How CLM Works

```
┌─────────────────────────────────────────────────────────────────────┐
│                    CLM: Contrastive Learning Approach                 │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   1. INPUT:                                                        │
│      • State: Full conversation history up to current turn            │
│      • Candidates: List of available tools                          │
│                                                                     │
│   2. PROCESSING:                                                   │
│      • Encode state → state embedding (Qwen3-8B)                   │
│      • Encode each tool → tool embedding (Qwen3-8B)               │
│      • Score: cosine_similarity(state_emb, tool_emb)                │
│                                                                     │
│   3. OUTPUT:                                                       │
│      • Selected tool with probability score                         │
│      • NOTE: CLM only selects tool, does NOT generate arguments    │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### Limitations

⚠️ **CLM chỉ chọn tool, không sinh arguments**

Để có complete function calling, cần kết hợp:
1. **CLM**: Chọn tool phù hợp (scoring)
2. **Generation Model**: Sinh arguments cho tool đã chọn

### References

- **CLM Model**: https://huggingface.co/Contrastive-LM/CLM-v0.1-8B
- **GitHub**: https://github.com/Contrastive-LM/CLM
- **Blog**: https://contrastive-lm.notion.site

---

*Report generated by CLM Evaluation Notebook (Turn-by-Turn)*  
*Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*
"""

display(Markdown(report_md))